# 37 · Securing A2A: auth schemes, extended cards and signed Agent Cards
Level L6 · Part 7 · professional · ~40 min · builds on 34, 09, 18

## Goal
By the end you protect triage-agent with an OAuth 2.0 `securitySchemes` entry and per-skill requirements. You serve an authenticated extended card to an admin only. You sign the public Agent Card with ES256 (JWS over RFC 8785 JSON) and verify it with a pinned key and a fixed algorithm list. A look-alike card is refused, and the orchestrator sends its task to the signed agent. The run makes 2 model calls. No Docker and no key are needed when the recording is there.

## Why it matters
Lessons 08, 09 and 18 gave triage-agent a card and a URL, and the orchestrator trusted both. Anyone who can put a card where the orchestrator looks becomes the agent. A look-alike host, a poisoned registry entry or a proxy in the middle can do it. The orchestrator then sends the issue text, which may hold customer details, to the impostor, and takes its answer as the truth. A card with no `securitySchemes` also says nothing about who may call the agent.

Without a check, each agent is one trust decision made on the URL alone. With one, the card names its scheme, its scopes and its signing key, and an unverified card is refused before any task is sent.

You don't need this for an agent on your own laptop or a card in your own config. It pays as soon as the card or the agent belongs to another team or host.

## The idea
An Agent Card is JSON, and anyone who can serve JSON can write one, so a card is a claim, not a proof. Three parts make it checkable. `securitySchemes` and `securityRequirements` say how to call the agent and with which scopes. An extended card holds details only an authenticated caller should see. A signature, made with a key the orchestrator already trusts, proves who made the card and that it did not change.

The verifier must not let the token or the card pick the algorithm or the key. So the notebook passes a fixed list, `["ES256"]`, and a pinned key store.

Analogy: a bank's letter of introduction. The letter says who you are and what you may do. A bank trusts it only when the stamp matches a seal it already holds, and the letter has not been altered since it was stamped.

## Picture
```mermaid
sequenceDiagram
  participant O as Orchestrator
  participant I as Look-alike (9138)
  participant A as triage-agent (9137)
  O->>I: GET agent card (public)
  I-->>O: card with no valid signature for a pinned key
  O-->>O: refuse the look-alike, nothing is sent
  O->>A: GET agent card (public)
  A-->>O: card signed with kid triage-card-1 (ES256)
  O-->>O: verify: pinned key, algorithms = ES256 only
  O->>A: SendMessage with a bearer token (aud = agent URL)
  A-->>O: artifact: the label
```

## Step by step

### 1. Set up the notebook
The ports are 9137 for triage-agent and 9138 for the look-alike. The issuer name, token URL and card key id come from `.env`. Their defaults point at the realm of lesson 34, which this lesson does not need to run.

In [1]:
import sys, json, time, asyncio, base64
from pathlib import Path
sys.path.insert(0, "..")                       # so llm_setup and llm_replay import
import httpx, uvicorn, jwt
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ec
import tut_env
from tut_env import env
import llm_replay
tut_env.load()
Path("../apps/37").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(Path("../apps/37").resolve()))
ISSUES = json.loads(Path("../data/trackr_05b_issues.json").read_text())
by_id = {i["id"]: i for i in ISSUES}
ISSUE = by_id[4]["title"]                     # the issue the orchestrator routes
AGENT_PORT, IMPOSTOR_PORT = 9137, 9138
AGENT_URL, IMPOSTOR_URL = f"http://127.0.0.1:{AGENT_PORT}/", f"http://127.0.0.1:{IMPOSTOR_PORT}/"
ISSUER = env("TRIAGE_ISSUER", "http://127.0.0.1:8340/realms/ops-desk")
TOKEN_URL = env("TRIAGE_TOKEN_URL", "http://127.0.0.1:8340/realms/ops-desk/protocol/openid-connect/token")
CARD_KID = env("TRIAGE_CARD_KID", "triage-card-1")
print("agent", AGENT_URL, "| look-alike", IMPOSTOR_URL, "| card key id", CARD_KID)

agent http://127.0.0.1:9137/ | look-alike http://127.0.0.1:9138/ | card key id triage-card-1


### 2. Without a check: the look-alike wins
A client that only reads cards takes the first card that claims the name. Here the look-alike is listed first. It has the same name and skill, and a new URL. Nothing in the card says it is not the real agent.

In [2]:
from a2a.types import AgentCard, AgentInterface, AgentSkill

def card_at(url: str, description: str) -> AgentCard:
    return AgentCard(name="triage-agent", description=description, version="1.1.0",
                     supported_interfaces=[AgentInterface(url=url, protocol_binding="JSONRPC", protocol_version="1.0")],
                     default_input_modes=["text/plain"], default_output_modes=["text/plain"],
                     skills=[AgentSkill(id="classify_issue", name="Classify an issue", tags=["triage"],
                                        description="Classifies a Trackr issue as bug, feature or question.")])

look_alike = card_at(IMPOSTOR_URL, "Classifies Trackr issues (new edition).")
listing = [look_alike]                                    # the look-alike is listed first
naive_pick = next(c for c in listing if c.name == "triage-agent")   # no checks at all
print("naive client sends the issue text to:", naive_pick.supported_interfaces[0].url)

naive client sends the issue text to: http://127.0.0.1:9138/


### 3. Declare what a caller needs
The public card now says how to get a token and which scopes exist. `triage:run` lets a caller classify. `triage:admin` lets a caller see the extended card. The scheme is OAuth 2.0 client credentials, so the orchestrator is an identity of its own, not a user. The skill `classify_issue` requires `triage:run`. The admin skill `relabel_batch` appears only on the extended card.

In [3]:
from a2a.types import (AgentCapabilities, ClientCredentialsOAuthFlow, OAuth2SecurityScheme, OAuthFlows,
                       SecurityRequirement, SecurityScheme, StringList)

def build_cards() -> tuple[AgentCard, AgentCard]:
    public = card_at(AGENT_URL, "Classifies Trackr issues, one label per issue. Needs the triage:run scope.")
    public.capabilities.CopyFrom(AgentCapabilities(streaming=False, push_notifications=False, extended_agent_card=True))
    public.security_schemes["org_oauth"].CopyFrom(SecurityScheme(oauth2_security_scheme=OAuth2SecurityScheme(
        description="The organisation's authorization server, client credentials grant.",
        flows=OAuthFlows(client_credentials=ClientCredentialsOAuthFlow(token_url=TOKEN_URL,
             scopes={"triage:run": "Classify issues", "triage:admin": "See the extended card"})))))
    public.security_requirements.append(SecurityRequirement(schemes={"org_oauth": StringList(list=["triage:run"])}))
    public.skills[0].security_requirements.append(SecurityRequirement(schemes={"org_oauth": StringList(list=["triage:run"])}))
    extended = AgentCard()
    extended.CopyFrom(public)
    extended.skills.append(AgentSkill(id="relabel_batch", name="Relabel a batch", tags=["admin"],
        description="Changes the label of many issues at once. Admins only.",
        security_requirements=[SecurityRequirement(schemes={"org_oauth": StringList(list=["triage:admin"])})]))
    return public, extended

public_card, extended_card = build_cards()
print("public skills:", [s.id for s in public_card.skills], "| extended skills:", [s.id for s in extended_card.skills])
print("scheme:", list(public_card.security_schemes), "| card requires:", list(public_card.security_requirements[0].schemes["org_oauth"].list))

public skills: ['classify_issue'] | extended skills: ['classify_issue', 'relabel_batch']
scheme: ['org_oauth'] | card requires: ['triage:run']


### 4. A lab issuer for the tokens
In production the orchestrator gets its token from the organisation's authorization server, as lesson 34 sets up. This lab cell makes a key pair in memory and signs tokens with it, so the lesson runs with no Docker. Only the public key is given to the agent. Each token names the audience, which is the one agent it is for.

In [4]:
def new_es256_pair() -> tuple[str, str]:
    key = ec.generate_private_key(ec.SECP256R1())
    private = key.private_bytes(serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8, serialization.NoEncryption())
    public = key.public_key().public_bytes(serialization.Encoding.PEM, serialization.PublicFormat.SubjectPublicKeyInfo)
    return private.decode(), public.decode()

ISSUER_PRIV, ISSUER_PUB = new_es256_pair()
def mint(scope: str, audience: str) -> str:
    now = int(time.time())
    claims = {"iss": ISSUER, "sub": "ops-orchestrator", "aud": audience, "scope": scope, "iat": now, "exp": now + 300}
    return jwt.encode(claims, ISSUER_PRIV, algorithm="ES256", headers={"kid": "lab-issuer-1"})

run_token = mint("triage:run", AGENT_URL)
claims = jwt.decode(run_token, ISSUER_PUB, algorithms=["ES256"], audience=AGENT_URL, issuer=ISSUER)
print("token claims:", {k: claims[k] for k in ("sub", "aud", "scope")})
try:
    jwt.decode(run_token, ISSUER_PUB, algorithms=["ES256"], audience=IMPOSTOR_URL, issuer=ISSUER)
    print("token accepted for the look-alike (this must not happen)")
except jwt.InvalidAudienceError:
    print("the same token is refused for the look-alike: wrong audience")

token claims: {'sub': 'ops-orchestrator', 'aud': 'http://127.0.0.1:9137/', 'scope': 'triage:run'}
the same token is refused for the look-alike: wrong audience


### 5. The agent server
The agent is a file, `securing_triage.py`. It has three new parts. `RequireToken` checks the bearer token on every request except the public card. `make_extended_modifier` decides what an authenticated caller gets on the extended card. `build_app` puts both around the lesson 09 executor. Read the file before you run it.

In [5]:
%%writefile ../apps/37/securing_triage.py
"""Lesson 37: triage-agent behind bearer tokens: a public card, an extended card for admins, and a token check."""
import jwt
from a2a.helpers import get_message_text, new_task_from_user_message, new_text_part
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import AgentCard
from starlette.applications import Starlette
from starlette.responses import JSONResponse

LLM = None                        # the notebook sets a real model here; None means the keyword rule

def bearer(header: str) -> str:
    return header[7:] if header.startswith("Bearer ") else ""

class TokenCheck:
    """Checks the lab issuer's ES256 tokens: signature with the pinned key, issuer, audience = this agent, expiry."""
    def __init__(self, issuer_pem: str, issuer: str, audience: str):
        self.key, self.issuer, self.audience = issuer_pem, issuer, audience

    def claims(self, token: str) -> dict | None:
        if not token:
            return None
        try:
            return jwt.decode(token, self.key, algorithms=["ES256"], issuer=self.issuer, audience=self.audience)
        except jwt.PyJWTError:
            return None           # wrong key, issuer, audience, or expired: no access

class RequireToken:
    """ASGI wrapper. Every request except the public card needs a token with the scope triage:run."""
    def __init__(self, app, check: TokenCheck):
        self.app, self.check = app, check

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http" or scope["path"] == "/.well-known/agent-card.json":
            return await self.app(scope, receive, send)
        header = dict(scope["headers"]).get(b"authorization", b"").decode()
        claims = self.check.claims(bearer(header)) or {}
        if "triage:run" not in claims.get("scope", "").split():
            reply = JSONResponse({"error": "invalid_token"}, status_code=401)
            return await reply(scope, receive, send)
        return await self.app(scope, receive, send)

def make_extended_modifier(check: TokenCheck):
    """Admins get the full extended card. Other callers with a valid token get the public skills only."""
    async def extended_card(card: AgentCard, context) -> AgentCard:
        claims = check.claims(bearer(context.state["headers"].get("authorization", ""))) or {}
        if "triage:admin" in claims.get("scope", "").split():
            return card
        public = AgentCard()           # a copy with the admin skill left out
        public.CopyFrom(card)
        del public.skills[:]
        public.skills.extend(s for s in card.skills if s.id != "relabel_batch")
        return public
    return extended_card

def scripted_label(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ("crash", "error", "fails", "does nothing")):
        return "bug"
    if any(w in t for w in ("add", "support", "would be nice")):
        return "feature"
    return "question"

class TriageExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        if context.current_task is None:
            await event_queue.enqueue_event(new_task_from_user_message(context.message))
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.start_work()
        text = get_message_text(context.message)
        if LLM is None:
            label, source = scripted_label(text), "scripted"
        else:
            reply = await LLM.ainvoke("Classify this Trackr issue as exactly one word: bug, feature or question.\nIssue: " + text)
            label, source = reply.text.strip().lower().split()[0].strip(".,:"), "model"
        await updater.add_artifact(parts=[new_text_part(label)], name=source)
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        await TaskUpdater(event_queue, context.task_id, context.context_id).cancel()

def build_app(public: AgentCard, extended: AgentCard, check: TokenCheck):
    handler = DefaultRequestHandler(agent_executor=TriageExecutor(), task_store=InMemoryTaskStore(),
                                    agent_card=public, extended_agent_card=extended,
                                    extended_card_modifier=make_extended_modifier(check))
    app = Starlette(routes=[*create_agent_card_routes(public), *create_jsonrpc_routes(handler, "/")])
    return RequireToken(app, check)

Overwriting ../apps/37/securing_triage.py


### 6. Sign the public card
The signer adds one signature to the card. The protected header names the key id and the algorithm. The signature covers the card's canonical JSON, with the `signatures` field left out, so the card cannot change after signing without the check failing. The notebook also makes the look-alike and signs it with a key of its own, using the same key id. That is the attacker's best try without the real key.

In [6]:
from a2a.utils.signing import create_agent_card_signer, create_signature_verifier
CARD_PRIV, CARD_PUB = new_es256_pair()           # the card key: in production it lives in a KMS, not in a notebook
ATTACKER_PRIV, ATTACKER_PUB = new_es256_pair()
HEADER = {"kid": CARD_KID, "alg": "ES256", "typ": "JOSE"}
signed_card = create_agent_card_signer(CARD_PRIV, HEADER)(public_card)
forged_look_alike = create_agent_card_signer(ATTACKER_PRIV, HEADER)(look_alike)

def protected_header(card: AgentCard) -> dict:
    part = card.signatures[0].protected
    return json.loads(base64.urlsafe_b64decode(part + "=" * (-len(part) % 4)))
print("signed card header:", protected_header(signed_card), "| signatures:", len(signed_card.signatures))

signed card header: {'alg': 'ES256', 'kid': 'triage-card-1', 'typ': 'JOSE'} | signatures: 1


### 7. Start the agent and the look-alike
The helper below serves an ASGI app as a task of this notebook, as in lesson 18. It waits until the card answers. triage-agent gets the signed card, the extended card and the token check. The look-alike serves only its card, as a fake site would.

In [7]:
class AgentServer:
    """One ASGI app served as a task of this notebook's event loop."""
    def __init__(self, app, port: int):
        self.url, self.server = f"http://127.0.0.1:{port}", uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))

    async def start(self, timeout: float = 15.0) -> "AgentServer":
        self.task = asyncio.create_task(self.server.serve())
        async with httpx.AsyncClient(timeout=1) as http:
            for _ in range(int(timeout / 0.2)):            # poll the card until it answers
                try:
                    if (await http.get(self.url + "/.well-known/agent-card.json")).status_code == 200:
                        return self
                except httpx.TransportError:
                    pass                                    # not up yet
                await asyncio.sleep(0.2)
        raise RuntimeError("the agent did not answer its card URL in time")

    async def stop(self) -> None:
        self.server.should_exit = True
        await self.task

from securing_triage import TokenCheck, build_app
from starlette.applications import Starlette
from a2a.server.routes import create_agent_card_routes
check = TokenCheck(ISSUER_PUB, ISSUER, audience=AGENT_URL)
agent = await AgentServer(build_app(signed_card, extended_card, check), AGENT_PORT).start()
impostor = await AgentServer(Starlette(routes=create_agent_card_routes(forged_look_alike)), IMPOSTOR_PORT).start()
print("up: triage-agent on", AGENT_PORT, "| look-alike on", IMPOSTOR_PORT)

up: triage-agent on 9137 | look-alike on 9138


### 8. Verify the card before you use it
The key provider reads only the pinned store, by key id. An unknown key id gives no key. The `jku` field in the header is ignored on purpose: it names a URL the attacker can choose. The verifier allows `ES256` only. Two cards are fetched through the same resolver. The real card passes, and the look-alike raises an error before any task is sent.

In [8]:
from a2a.client import A2ACardResolver
TRUSTED = {CARD_KID: CARD_PUB}                   # the pinned store: key id -> public key
ALGORITHMS = ["ES256"]

def trusted_key(kid, jku):
    if kid not in TRUSTED:
        raise jwt.InvalidKeyError(f"unknown key id {kid!r}")
    return TRUSTED[kid]                          # jku is never fetched: a card may not choose its own key

verify_card = create_signature_verifier(trusted_key, ALGORITHMS)
async with httpx.AsyncClient(timeout=10) as http:
    resolver = A2ACardResolver(http, AGENT_URL)
    verified = await resolver.get_agent_card(signature_verifier=verify_card)
    print("real card verified:", verified.name, "| url", verified.supported_interfaces[0].url)
    try:
        await A2ACardResolver(http, IMPOSTOR_URL).get_agent_card(signature_verifier=verify_card)
        print("look-alike accepted (this must not happen)")
    except Exception as e:
        print("look-alike refused:", type(e).__name__, "-", e)

real card verified: triage-agent | url http://127.0.0.1:9137/
look-alike refused: InvalidSignaturesError - No valid signature found


### 9. Other forgeries the verifier refuses
Five ways to forge or change a card, each refused. The HS256 case is the classic trap: the attacker sets `alg` to HS256 and uses the pinned public key as the HMAC secret. The allow-list refuses it first. The token's `alg` is HS256, the list holds only ES256, so PyJWT raises before it looks at any key. PyJWT's key check is a second line of defence: it refuses a PEM public key as an HMAC secret, but only when HS256 is on the list. The table shows the verifier's wrapper error, `InvalidSignaturesError`. The next cell shows PyJWT's own errors for the same token.

In [9]:
from a2a.types import AgentCardSignature
def attempt(label: str, card: AgentCard, verifier=verify_card) -> None:
    try:
        verifier(card)
        print(f"{label:34} ACCEPTED")
    except Exception as e:
        print(f"{label:34} refused: {type(e).__name__}")

attempt("unsigned copy", AgentCard(name="triage-agent"))
tampered = AgentCard()
tampered.CopyFrom(signed_card)
tampered.supported_interfaces[0].url = IMPOSTOR_URL      # change one field after signing
attempt("signed card, URL changed", tampered)
fresh = card_at(IMPOSTOR_URL, "Classifies Trackr issues (new edition).")
attempt("signed by an unknown key id", create_agent_card_signer(ATTACKER_PRIV, {**HEADER, "kid": "attacker-1"})(fresh))
hs_forged = AgentCard(name="triage-agent")
hs_header = base64.urlsafe_b64encode(json.dumps({"kid": CARD_KID, "alg": "HS256"}).encode()).rstrip(b"=").decode()
hs_forged.signatures.append(AgentCardSignature(protected=hs_header, signature="AAAA"))
attempt("HS256 forgery, pinned key", hs_forged)
attempt("look-alike signed by its own key", forged_look_alike)   # the card the impostor really serves

unsigned copy                      refused: NoSignatureError
signed card, URL changed           refused: InvalidSignaturesError
signed by an unknown key id        refused: InvalidSignaturesError
HS256 forgery, pinned key          refused: InvalidSignaturesError
look-alike signed by its own key   refused: InvalidSignaturesError


The HS256 token from the table, decoded directly with PyJWT. The HMAC is keyed with the pinned public PEM, as an attacker would do it. First the policy list `["ES256"]`, then the same list with `HS256` added.

In [10]:
import hmac, hashlib
def b64url(raw: bytes) -> str:
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode()
hs_header = b64url(json.dumps({"alg": "HS256", "typ": "JWT", "kid": CARD_KID}).encode())
hs_body = b64url(json.dumps({"sub": "attacker", "aud": AGENT_URL, "scope": "triage:run triage:admin", "exp": int(time.time()) + 300}).encode())
hs_sig = b64url(hmac.new(CARD_PUB.encode(), f"{hs_header}.{hs_body}".encode(), hashlib.sha256).digest())
hs_token = f"{hs_header}.{hs_body}.{hs_sig}"
for algs in (["ES256"], ["ES256", "HS256"]):
    try:
        jwt.decode(hs_token, CARD_PUB, algorithms=algs)
        print(algs, "accepted (this must not happen)")
    except jwt.PyJWTError as e:
        print(algs, "->", type(e).__name__, "-", e)

['ES256'] -> InvalidAlgorithmError - The specified alg value is not allowed
['ES256', 'HS256'] -> InvalidKeyError - The specified key is an asymmetric key or x509 certificate and should not be used as an HMAC secret.


### 10. The authenticated extended card
Two clients come from the verified card. The one with a token sends the token with each call, through `AuthInterceptor`, so the scheme in the card decides how the token goes out. An anonymous call is refused at the door. A caller with `triage:run` gets the public skills. An admin gets the extended card with `relabel_batch`.

In [11]:
from a2a.client import ClientCallContext, ClientConfig, create_client
from a2a.client.auth.credentials import InMemoryContextCredentialStore
from a2a.client.auth.interceptor import AuthInterceptor
from a2a.types import GetExtendedAgentCardRequest

async def extended_for(scope: str) -> list[str]:
    store = InMemoryContextCredentialStore()
    await store.set_credentials("ops", "org_oauth", mint(scope, AGENT_URL))
    client = await create_client(agent=verified, client_config=ClientConfig(streaming=False),
                                 interceptors=[AuthInterceptor(store)])
    card = await client.get_extended_agent_card(GetExtendedAgentCardRequest(), context=ClientCallContext(state={"sessionId": "ops"}))
    await client.close()
    return [s.id for s in card.skills]

anonymous = await create_client(agent=verified, client_config=ClientConfig(streaming=False))
try:
    await anonymous.get_extended_agent_card(GetExtendedAgentCardRequest())
    print("anonymous: got the card (this must not happen)")
except Exception as e:
    print("anonymous: refused:", type(e).__name__)
await anonymous.close()
print("triage:run  sees skills:", await extended_for("triage:run"))
print("triage:admin sees skills:", await extended_for("triage:run triage:admin"))

anonymous: refused: A2AClientError
triage:run  sees skills: ['classify_issue']
triage:admin sees skills: ['classify_issue', 'relabel_batch']


### 11. The orchestrator routes and delegates (`needs-llm`)
The orchestrator has the verified card and a token. It asks the model one question: which skill from the verified card fits this issue? That is the orchestrator's model call. Then it sends the issue to triage-agent with the token. The agent makes one model call to classify it. The notebook plans two calls in total. If the model cell is skipped, a fixed skill id and the keyword rule stand in, and the labels say so.

In [12]:
llm_replay.start("37_securing-a2a")
from llm_setup import get_llm
llm = get_llm(max_calls=2)
import securing_triage
securing_triage.LLM = llm                       # the agent classifies with the same model: call 2
SKILLS = "\n".join(f"{s.id}: {s.description}" for s in verified.skills)
reply = await llm.ainvoke(f"You route Trackr work. Skills:\n{SKILLS}\nIssue: {ISSUE}\nReply with the skill id only.")
chosen = reply.text.strip().split()[0].strip(".,:`'\"")
print("orchestrator model chose:", chosen)

LLM mode: replay (37_securing-a2a.yaml)


Gemini on Vertex AI · API key loaded: True
orchestrator model chose: classify_issue


If the model cell was skipped, the keyword rule and a fixed skill id take over. If the model cell ran, this cell prints nothing.

In [13]:
if "llm" not in globals() or llm is None:
    chosen = "classify_issue"
    print("no model here: the orchestrator uses the fixed skill id classify_issue (stand-in)")
    print("the agent uses the keyword rule (artifact name 'scripted')")
assert chosen in [s.id for s in verified.skills], f"{chosen!r} is not a skill on the verified card"

The orchestrator sends the issue with its token. The request goes to the verified card's URL. The reply is the artifact, which is the answer for the user. The orchestrator does not send it back to the model.

In [14]:
from a2a.helpers import get_artifact_text, new_text_message
from a2a.types import Role, SendMessageRequest, TaskState
creds = InMemoryContextCredentialStore()
await creds.set_credentials("ops", "org_oauth", mint("triage:run", AGENT_URL))
orchestrator = await create_client(agent=verified, client_config=ClientConfig(streaming=False),
                                   interceptors=[AuthInterceptor(creds)])
task = None
request = SendMessageRequest(message=new_text_message(ISSUE, role=Role.ROLE_USER))
async for chunk in orchestrator.send_message(request, context=ClientCallContext(state={"sessionId": "ops"})):
    if chunk.WhichOneof("payload") == "task":
        task = chunk.task
print("task state:", TaskState.Name(task.status.state), "| skill:", chosen)
print("artifact:", get_artifact_text(task.artifacts[0]), "| name:", task.artifacts[0].name)

task state: TASK_STATE_COMPLETED | skill: classify_issue
artifact: bug | name: model


### 12. Stop the servers
The clients and servers stop, and the recording closes. The model calls are all made before this cell.

In [15]:
await orchestrator.close()
await impostor.stop()
await agent.stop()
llm_replay.stop()
print("lesson 37 done: the card was verified before use, and only the signed agent got the task")

lesson 37 done: the card was verified before use, and only the signed agent got the task


## At real scale
- **Key rotation.** Publish the new card key before you sign with it. Keep the old key in the trust store until every card signed with it has expired. Keycloak's guide ("Rotating keys", https://www.keycloak.org/docs/latest/server_admin/index.html) gives the same order.
- **Trust store as code.** The pinned keys live in reviewed configuration, not in the card. A key provider that fetches a `jku` URL lets the card name its own key, so keep it off.
- **Scopes per skill, not per server.** Each skill's `securityRequirements` tells a client what to ask for. The server still checks the token on every call. Declaring a requirement does not enforce it.
- **Errors do not leak.** An anonymous caller learns only that access is refused. Do not reveal which skills or tenants exist. → lesson 38 for tenant isolation.

## Decision guide
| The question | Answer |
|---|---|
| Is the agent inside one team's network? | A bearer token may be enough. |
| Does the card cross a team or company boundary? | Sign the card and pin its key. |
| Does a skill expose more than the public skills? | Put it on the extended card and require a scope. |
| Does a token name one audience only? | Yes, always. Check `aud` on the server. |
| Does the verifier allow one algorithm? | Yes. Pass `algorithms=[...]` from your own policy, never from the token. |

## What just happened
The look-alike was never sent a task. Its card failed the signature check, and the orchestrator used the signed card. The token then limited who could call the agent and which skills they could see. Three controls worked together. The card signature answers "is this the agent I trust?". The token answers "may this caller use it?". The extended card answers "what may this caller see?".

The notebook signed the public card. An extended card is served only over the authenticated call, so the token protects it. Sign it too when a copy of the extended card can leave the authenticated path.

## Common mistakes
**1. Signing, then changing the card.** A card changed after signing fails the check. Here the URL changes after the signature. The verifier raises, and the cell stops with the error.

In [16]:
verify_card(tampered)      # wrong: the card was changed after signing, so this must fail

InvalidSignaturesError: No valid signature found

The fix: change the card first, sign it last, and then publish it. Re-sign after every change.

**2. Letting the token pick the algorithm.** The allow-list is policy, not something the token's header may set. If the list is built from the token's `alg` (`algorithms=[alg]`), the attacker chooses the algorithm. If `HS256` sits next to `ES256`, the algorithm check passes and only PyJWT's key check stops the forgery. With `["ES256"]` alone, the HS256 token fails at the algorithm check, as the next error shows.

In [17]:
jwt.decode(hs_token, CARD_PUB, algorithms=["ES256"])      # the HS256 token from step 9: the policy list refuses it

InvalidAlgorithmError: The specified alg value is not allowed

The fix: pass `ALGORITHMS` from your own policy, as step 8 does. Never build the list from the token.

**3. Trusting `jku`.** A card that names a key URL lets the author choose the key. Fetch keys only from a pinned store.

## Try it
Change the description of `signed_card` and verify it. It must be refused. Then sign the changed card again with `CARD_PRIV` and verify it. It must pass. Print the two results. Use `attempt` from step 9 and `create_agent_card_signer` from step 6.

In [18]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [19]:
changed = AgentCard()
changed.CopyFrom(signed_card)
changed.description = "Classifies Trackr issues, and keeps a copy of each one."
attempt("changed, still signed by the old sig", changed)
resigned = create_agent_card_signer(CARD_PRIV, HEADER)(changed)
attempt("changed and signed again", resigned)

changed, still signed by the old sig refused: InvalidSignaturesError
changed and signed again           ACCEPTED


## Key terms
- **Agent Card signature** — a JWS over the card's RFC 8785 canonical JSON, with the `signatures` field left out. It proves the card came from the key holder and did not change.
- **extended Agent Card** — a card with more skills, served only to authenticated callers through `GetExtendedAgentCard`.
- **securitySchemes and securityRequirements** — the card's list of ways to authenticate (here OAuth 2.0 client credentials) and which scopes each skill needs.
- **pinned key store** — the verifier's own map from key id to public key. The card cannot change it.
- **algorithm allow-list** — the fixed list of signing algorithms a verifier accepts, chosen by policy, never taken from the token.
- **audience (`aud`)** — the one resource a token is for. The agent refuses a token meant for another audience.